# Hugging Face Code APIs: High-Level vs. Low-Level

When writing Python code using the `transformers` library, Hugging Face gives you two different "APIs" (ways to interact with the code) to run models locally.

1. **The High-Level API (`pipeline`)**: The "easy button" that does everything for you in one line of code.
2. **The Low-Level API (`AutoClasses`)**: The manual transmission. You write out every mathematical step yourself.

Understanding the difference is crucial for debugging and building advanced, custom AI architectures.

## 1. The High-Level API: `pipeline()`

**What it is:** The `pipeline` function is a massive abstraction wrapper. It is designed so that anyone can use AI without knowing how neural networks actually function.

**What it hides from you:**
When you pass English text to a pipeline, it secretly does three things:
1. **Pre-processing:** It converts your English words into a mathematical tensor (an array of numbers) because GPUs cannot read English.
2. **Forward Pass:** It shoves those numbers through the neural network on your GPU.
3. **Post-processing:** It takes the raw probability numbers the model spits out and converts them back into readable English.

In [2]:
from transformers import pipeline

print("--- Using the High-Level Pipeline ---")

# 1. We load a basic sentiment analysis model directly onto our RTX 4080 (device=0)
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", device=0)

# 2. We just pass raw English! The pipeline handles all the math.
result = classifier("I am absolutely loving learning about AI architecture!")

print(result)

--- Using the High-Level Pipeline ---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998714923858643}]


## 2. The Low-Level API: AutoClasses

**What it is:** This is how actual AI Engineers write code. Instead of using a pipeline, we manually invoke the two core components of every Language Model:
1. **The Tokenizer:** The dictionary that translates text to numbers.
2. **The Model:** The neural network weights.

To do this, Hugging Face provides `AutoTokenizer` and `AutoModel`. We have to write the code for the pre-processing, the GPU memory transfer, and the post-processing ourselves.

**Why use this instead of pipelines?**
* **Total Control:** You can alter exactly how the text is chopped up, manage how the GPU memory is allocated, and extract hidden layers from the neural network that the pipeline usually throws away.
* **Batching:** If you are processing 10,000 rows of a CSV, writing a custom low-level loop is significantly faster and more memory-efficient than using a pipeline.

In [3]:
# [PYTHON CELL]
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

print("--- Using the Low-Level AutoClasses ---")
model_id = "distilbert-base-uncased-finetuned-sst-2-english"

# 1. Load the Tokenizer and the Model manually
tokenizer = AutoTokenizer.from_pretrained(model_id)
# We manually send the model to the GPU using standard PyTorch syntax
model = AutoModelForSequenceClassification.from_pretrained(model_id).to("cuda")

raw_text = "I am absolutely loving learning about AI architecture!"

# STEP 1: PRE-PROCESSING (Tokenization)
# We convert the text to numbers and format it as a PyTorch tensor ("pt"), sending it to the GPU
inputs = tokenizer(raw_text, return_tensors="pt").to("cuda")
print(f"Tokenized Math: {inputs['input_ids']}")

# STEP 2: FORWARD PASS (Inference)
# We tell PyTorch not to calculate gradients (saves massive memory since we aren't training)
with torch.no_grad():
    outputs = model(**inputs)

# STEP 3: POST-PROCESSING (Decoding the math)
# The model outputs raw "logits" (unnormalized scores). We use a softmax function to turn them into percentages.
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(f"Raw Output Probabilities: {predictions}")

# Determine the winning label (0 = Negative, 1 = Positive)
predicted_class_id = predictions.argmax().item()
label = model.config.id2label[predicted_class_id]

print(f"Final Human Result: {{'label': '{label}'}}")

--- Using the Low-Level AutoClasses ---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Tokenized Math: tensor([[ 101, 1045, 2572, 7078, 8295, 4083, 2055, 9932, 4294,  999,  102]],
       device='cuda:0')
Raw Output Probabilities: tensor([[1.2844e-04, 9.9987e-01]], device='cuda:0')
Final Human Result: {'label': 'POSITIVE'}


## Quick Reference

| Feature | `pipeline()` | `AutoTokenizer` & `AutoModel` |
| :--- | :--- | :--- |
| **Complexity** | Extremely Easy (1 line of code) | Complex (Requires PyTorch knowledge) |
| **Customization** | Very Low | Maximum |
| **Visibility** | Black Box (Input text -> Output text) | Transparent (You see the tensors and logits) |
| **Best For** | Quick prototyping, simple tasks | Production apps, custom data pipelines, research |